# Risk-first trading assistant: Claude proposes, code enforces

A common first attempt at an LLM trading assistant is a long system prompt full of rules ("risk at most 1% per trade", "always set a stop") plus a JSON output format. The prompt is a good start, but it isn't a control. A model can misread a number, get the sizing arithmetic wrong, or talk itself into a trade that breaks a rule. When the rule is a hard limit on capital, you need an enforcement layer the model can't talk its way past.

This cookbook takes such a prompt and wraps it in a pipeline where code computes every number Claude sees, and code re-checks every trade Claude proposes before it could reach a broker:

```
OHLCV ─► indicators + data checks ─► circuit breakers ─► Claude ─► risk engine ─► orders + audit log
          (code)                      (code)             (JSON)     (code)
                                        │ limit breached   │ API error, refusal, bad output
                                        └──────────────────┴─► fail closed: NO_TRADE
```

Claude handles judgment: reading the data-quality report, weighing scheduled events and headlines, and writing a reason for each decision that a human can audit. Code handles arithmetic and hard limits. The risk engine may shrink a trade or reject it. It never makes a trade bigger, never loosens a stop, and never moves a profit target to make a trade pass.

> **Not financial advice.** All market data, events, and headlines in this notebook are synthetic, and nothing here connects to a broker. Treat it as a template for paper trading, not a production trading system.

## What you'll learn

- How to turn a prompt template with `[placeholders]` into one config object that both the prompt and the enforcement code read, so they can't drift apart
- How to compute indicators and sanity-check data before the model sees them, so Claude never has to invent or recompute a number
- How to guarantee the response shape with structured outputs (`client.messages.parse` and a Pydantic model)
- How to build a risk engine that resizes, rejects, and force-exits in code, and fails closed when the model is unavailable
- How to keep an audit trail a human can review

## Prerequisites

- Python 3.11+
- An Anthropic API key in a `.env` file (`ANTHROPIC_API_KEY=...`). Without one, the deterministic steps still run and the model step fails closed to `NO_TRADE`.
- Basic familiarity with OHLCV bars, EMAs, RSI, and ATR

## Setup

In [1]:
%pip install -q anthropic numpy pandas pydantic python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import hashlib
import json
import math
from dataclasses import asdict, dataclass, field
from typing import Literal
from zoneinfo import ZoneInfo

import anthropic
import numpy as np
import pandas as pd
import pydantic
from dotenv import load_dotenv
from pandas.tseries.holiday import USFederalHolidayCalendar
from pandas.tseries.offsets import CustomBusinessDay
from pydantic import BaseModel

load_dotenv()
client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-6"
ET = ZoneInfo("America/New_York")

## 1. Fill in the template with a config object

The original prompt has placeholders like `[1%]` and `[3]`. If you paste values straight into the prompt text, the limits live in one place for the model and somewhere else (or nowhere) for your code. Put them in one frozen dataclass instead: the system prompt is rendered from it, and the risk engine reads the same fields.

The values below are the prompt's own examples: US stocks on 1-hour bars, long only, 1% risk per trade, stops at 1.5 × ATR, and a 2:1 minimum reward:risk. The last group holds data-sanity limits that the prompt implies ("if data is missing or looks wrong") but doesn't quantify.

In [3]:
@dataclass(frozen=True)
class TradingConfig:
    # Market & universe
    asset_class: str = "US stocks"
    symbols: tuple[str, ...] = ("SPY", "QQQ", "AAPL", "MSFT")
    timeframe: str = "1-hour candles, swing trades held 2-10 days"
    trading_hours: str = "US regular session only (09:30-16:00 ET)"
    session_open: str = "09:30"
    session_close: str = "16:00"

    # Strategy
    rsi_min: float = 40.0
    rsi_max: float = 65.0
    event_blackout_minutes: int = 30
    allow_shorts: bool = False

    # Hard risk limits
    risk_per_trade_pct: float = 1.0
    atr_stop_multiple: float = 1.5
    min_reward_risk: float = 2.0
    max_open_positions: int = 3
    max_position_pct: float = 20.0
    daily_loss_limit_pct: float = 3.0
    max_drawdown_pct: float = 10.0
    max_leverage: float = 1.0  # 1.0 means no leverage

    # Data sanity, enforced in code
    min_bars: int = 60  # enough history to warm up the 50 EMA
    max_bar_age_minutes: int = 90
    max_entry_deviation_pct: float = 0.5  # entry must be within 0.5% of the last close
    fractional_quantity: bool = False


CONFIG = TradingConfig()

The system prompt below is the original with each placeholder filled from `CONFIG`. A few lines are new. Each one closes a gap that would otherwise leave the model guessing:

- **Indicators arrive precomputed**, and the model is told not to recompute them.
- **"Crosses above" is defined**: at or below on the previous bar, above on the latest bar.
- **`SELL` vs. `CLOSE`.** With shorts disabled, `SELL` is ambiguous: exit a long, or open a short? Here, exits use `CLOSE` and `SELL` only ever means a short.
- **Exactly one action per symbol**, so every symbol gets a decision you can audit.
- **The risk engine is disclosed.** Claude knows it exists and is told not to lean on it.

In [4]:
def render_system_prompt(cfg: TradingConfig) -> str:
    if cfg.allow_shorts:
        shorts_rule = "Shorts are allowed under the same rules, mirrored."
    else:
        shorts_rule = (
            "No shorts. SELL means opening a short, so never use it; exit a long with CLOSE."
        )
    leverage_rule = (
        "No leverage." if cfg.max_leverage <= 1 else f"Leverage up to {cfg.max_leverage:g}x."
    )

    return f"""ROLE
You are a disciplined, risk-first trading assistant. You analyze market data you are given and propose trades. You do not predict the future with certainty, and your first priority is protecting capital. Your second priority is steady, repeatable returns.

MARKET & UNIVERSE
- Asset class: {cfg.asset_class}
- Tradable symbols: {", ".join(cfg.symbols)}
- Timeframe: {cfg.timeframe}
- Trading hours: {cfg.trading_hours}

INPUTS YOU WILL RECEIVE
Each cycle you'll get: current account balance, open positions, recent OHLCV price data, indicators (RSI 14, 20/50 EMA, ATR 14, volume and its 20-period average) for the latest and previous bar, a data-quality report per symbol, scheduled events, and news headlines. The indicators are computed by the data pipeline: use them as given and do not recompute them. Use ONLY the data provided. Never invent prices, news, or indicator values. If data is missing or looks wrong, return NO_TRADE for that symbol and explain why.

STRATEGY
- Long entry: price above the 50 EMA, 20 EMA crosses above the 50 EMA (at or below it on the previous bar, above it on the latest bar), RSI between {cfg.rsi_min:g} and {cfg.rsi_max:g}, volume above its 20-period average.
- Exit: close below the 20 EMA, or stop-loss / take-profit hit.
- {shorts_rule}
- Skip new entries within {cfg.event_blackout_minutes} minutes of major scheduled events (earnings, CPI, FOMC).

RISK RULES (HARD LIMITS, NEVER OVERRIDE)
- Risk at most {cfg.risk_per_trade_pct:g}% of account equity per trade. Size = (equity × risk%) ÷ (entry − stop), then cap it at the position limit below and round down to whole shares.
- Every trade must have a stop-loss, set at {cfg.atr_stop_multiple:g} × ATR below entry.
- Minimum reward:risk of {cfg.min_reward_risk:g}:1.
- Max {cfg.max_open_positions} open positions and max {cfg.max_position_pct:g}% of equity in any one position.
- If daily loss reaches {cfg.daily_loss_limit_pct:g}% of equity, stop trading for the day.
- If drawdown from peak reaches {cfg.max_drawdown_pct:g}%, stop trading and flag for human review.
- Never average down on a losing position. {leverage_rule}

DECISION PROCESS
1. Check the risk limits first. If any is breached, output NO_TRADE.
2. Evaluate each symbol against the strategy rules.
3. Only act when ALL entry conditions are met. When in doubt, don't trade.
4. Review open positions for exit conditions.

OUTPUT
Return exactly one action per tradable symbol:
- BUY: a new long entry, with every numeric field filled in.
- HOLD: an open position with no exit condition met. Repeat its quantity, stop_loss, and take_profit.
- CLOSE: exit an open position. quantity is the full position, order_type is MARKET, entry_price is the latest close.
- NO_TRADE: no position and no valid setup. Set the numeric fields to 0.
- confidence: if your confidence in a BUY would be LOW, don't take it.
- reasoning: one or two sentences that cite the specific conditions met or failed, with their values.
- timestamp: the as_of time you were given.
- warnings: anything a human reviewer should know, such as data problems, nearby events, or limits close to breach.

An independent risk engine re-checks every proposal after you. It rejects or shrinks anything that breaks a hard limit, and it never makes a trade bigger. Don't lean on it: propose only trades you believe pass every rule.

BEHAVIOR
- Be conservative. Doing nothing is a valid and often correct choice.
- Don't chase losses or "make back" money after a losing trade.
- Log clear reasoning so a human can audit every decision."""


SYSTEM_PROMPT = render_system_prompt(CONFIG)
print(SYSTEM_PROMPT)

ROLE
You are a disciplined, risk-first trading assistant. You analyze market data you are given and propose trades. You do not predict the future with certainty, and your first priority is protecting capital. Your second priority is steady, repeatable returns.

MARKET & UNIVERSE
- Asset class: US stocks
- Tradable symbols: SPY, QQQ, AAPL, MSFT
- Timeframe: 1-hour candles, swing trades held 2-10 days
- Trading hours: US regular session only (09:30-16:00 ET)

INPUTS YOU WILL RECEIVE
Each cycle you'll get: current account balance, open positions, recent OHLCV price data, indicators (RSI 14, 20/50 EMA, ATR 14, volume and its 20-period average) for the latest and previous bar, a data-quality report per symbol, scheduled events, and news headlines. The indicators are computed by the data pipeline: use them as given and do not recompute them. Use ONLY the data provided. Never invent prices, news, or indicator values. If data is missing or looks wrong, return NO_TRADE for that symbol and expla

## 2. Define the output contract

The prompt's JSON format becomes a Pydantic model. When you pass it to `client.messages.parse()`, the SDK sends it as a structured-outputs schema, so the response is guaranteed to parse into a `TradingDecision` with valid enum values. You don't need regex extraction or a "JSON only, no extra text" instruction.

Structured outputs guarantee the *shape*, not the *content*. A perfectly formed object can still carry a quantity ten times too large. Catching that is the risk engine's job in step 5.

In [5]:
class AccountStatus(BaseModel):
    equity: float
    daily_pnl_pct: float
    drawdown_pct: float


class TradeAction(BaseModel):
    symbol: str
    action: Literal["BUY", "SELL", "HOLD", "CLOSE", "NO_TRADE"]
    quantity: float
    order_type: Literal["LIMIT", "MARKET"]
    entry_price: float
    stop_loss: float
    take_profit: float
    risk_amount: float
    confidence: Literal["LOW", "MEDIUM", "HIGH"]
    reasoning: str


class TradingDecision(BaseModel):
    timestamp: str
    account_status: AccountStatus
    actions: list[TradeAction]
    warnings: list[str]

## 3. Build market data, indicators, and data checks

Everything in this step is deterministic. The notebook generates synthetic 1-hour bars with fixed seeds, each tuned to exercise a different branch of the logic:

| Symbol | Setup on the latest bar |
|---|---|
| AAPL | Fresh 20/50 EMA golden cross; all four entry conditions hold |
| MSFT | Fresh golden cross, but RSI is above 65 (a near miss) |
| SPY | Downtrend below the 50 EMA |
| QQQ | Held position; the bar closed below the 20 EMA (exit signal) |

In production, replace `make_bars` with your data feed. Nothing downstream changes.

In [6]:
BAR_OPENS = ["09:30", "10:30", "11:30", "12:30", "13:30", "14:30", "15:30"]
BAR_DURATION = pd.Timedelta(hours=1)  # the 15:30 bar is really 30 minutes; ignored here
TRADING_DAY = CustomBusinessDay(
    calendar=USFederalHolidayCalendar()
)  # close enough to NYSE for a demo

LAST_BAR_OPEN = pd.Timestamp("2026-09-25 14:30", tz=ET)  # last completed bar
NOW = pd.Timestamp("2026-09-25 15:35", tz=ET)  # when this cycle runs


def bar_index(n_bars: int, last_bar_open: pd.Timestamp) -> pd.DatetimeIndex:
    days = pd.date_range(
        end=last_bar_open.normalize(), periods=n_bars // len(BAR_OPENS) + 2, freq=TRADING_DAY
    )
    stamps = [pd.Timestamp(f"{d:%Y-%m-%d} {t}", tz=ET) for d in days for t in BAR_OPENS]
    stamps = [s for s in stamps if s <= last_bar_open]
    return pd.DatetimeIndex(stamps[-n_bars:])


def make_bars(seed, start_price, segments, base_volume, n_bars):
    # Random-walk OHLCV. `segments` is a list of (bars, drift per bar, volatility per bar).
    rng = np.random.default_rng(seed)
    returns = np.concatenate([rng.normal(drift, vol, n) for n, drift, vol in segments])
    close = start_price * np.cumprod(1 + returns)
    open_ = np.r_[start_price, close[:-1]] * (1 + rng.normal(0, 0.001, len(close)))
    high = np.maximum(open_, close) * (1 + np.abs(rng.normal(0, 0.002, len(close))))
    low = np.minimum(open_, close) * (1 - np.abs(rng.normal(0, 0.002, len(close))))
    volume = (base_volume * rng.lognormal(0, 0.25, len(close))).round(-2)
    bars = pd.DataFrame({"open": open_, "high": high, "low": low, "close": close, "volume": volume})
    bars = bars.iloc[:n_bars]
    bars.index = bar_index(n_bars, LAST_BAR_OPEN)
    return bars


MARKET = {
    "AAPL": make_bars(1, 232, [(70, -0.0010, 0.004), (60, 0.0015, 0.004)], 900_000, n_bars=99),
    "MSFT": make_bars(24, 505, [(70, -0.0010, 0.004), (60, 0.0030, 0.004)], 700_000, n_bars=105),
    "SPY": make_bars(7, 575, [(100, -0.0008, 0.003)], 5_000_000, n_bars=100),
    "QQQ": make_bars(28, 480, [(90, 0.0012, 0.0035), (10, -0.0025, 0.003)], 3_000_000, n_bars=100),
}
MARKET["AAPL"].tail(3).round(2)

,open,high,low,close,volume
2026-09-25 12:30:00-04:00,219.02,219.87,218.58,219.51,846800.0
2026-09-25 13:30:00-04:00,219.62,220.33,219.06,219.86,1080300.0
2026-09-25 14:30:00-04:00,219.61,219.86,218.88,218.94,1077700.0


Next come the indicators, the data-quality checks, and a compact per-symbol snapshot. The snapshot is the only market data Claude sees. The risk engine reads the same snapshot, so the model and the code judge the same rounded numbers.

In [7]:
def add_indicators(bars: pd.DataFrame) -> pd.DataFrame:
    out = bars.copy()
    out["ema20"] = out["close"].ewm(span=20, adjust=False).mean()
    out["ema50"] = out["close"].ewm(span=50, adjust=False).mean()
    change = out["close"].diff()
    avg_gain = change.clip(lower=0).ewm(alpha=1 / 14, adjust=False).mean()
    avg_loss = (-change.clip(upper=0)).ewm(alpha=1 / 14, adjust=False).mean()
    out["rsi14"] = 100 - 100 / (1 + avg_gain / avg_loss)
    prev_close = out["close"].shift()
    true_range = pd.concat(
        [
            out["high"] - out["low"],
            (out["high"] - prev_close).abs(),
            (out["low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)
    out["atr14"] = true_range.ewm(alpha=1 / 14, adjust=False).mean()
    out["volume_sma20"] = out["volume"].rolling(20).mean()
    return out


def check_data_quality(bars: pd.DataFrame, now: pd.Timestamp, cfg: TradingConfig) -> list[str]:
    issues = []
    if len(bars) < cfg.min_bars:
        issues.append(f"only {len(bars)} bars, need {cfg.min_bars}")
    recent = bars.tail(20)
    if recent.isna().any().any():
        issues.append("missing values in recent bars")
    if (recent[["open", "high", "low", "close"]] <= 0).any().any():
        issues.append("non-positive prices in recent bars")
    body_high = recent[["open", "close"]].max(axis=1)
    body_low = recent[["open", "close"]].min(axis=1)
    bad_range = (recent["high"] < body_high) | (recent["low"] > body_low)
    if bad_range.any():
        issues.append(
            f"{int(bad_range.sum())} recent bar(s) where high/low don't contain open/close"
        )
    if (recent["volume"] <= 0).any():
        issues.append("zero volume in recent bars")
    age_minutes = (now - (bars.index[-1] + BAR_DURATION)).total_seconds() / 60
    if age_minutes > cfg.max_bar_age_minutes:
        issues.append(f"stale data: last bar closed {age_minutes:.0f} min ago")
    return issues


def _num(value, digits=2):
    return None if pd.isna(value) else round(float(value), digits)


def build_snapshot(symbol: str, bars: pd.DataFrame, now, cfg: TradingConfig) -> dict:
    ind = add_indicators(bars)

    def indicator_row(i):
        row = ind.iloc[i]
        return {
            "bar_open": ind.index[i].isoformat(),
            "close": _num(row["close"]),
            "ema20": _num(row["ema20"]),
            "ema50": _num(row["ema50"]),
            "rsi14": _num(row["rsi14"]),
            "atr14": _num(row["atr14"]),
            "volume": _num(row["volume"], 0),
            "volume_sma20": _num(row["volume_sma20"], 0),
        }

    recent_bars = [
        {"bar_open": ts.isoformat(), **{k: _num(v) for k, v in row.items()}}
        for ts, row in bars.tail(5).iterrows()
    ]
    issues = check_data_quality(bars, now, cfg)
    return {
        "symbol": symbol,
        "recent_bars": recent_bars,
        "indicators": {"latest": indicator_row(-1), "previous": indicator_row(-2)},
        "data_quality": {"ok": not issues, "issues": issues},
    }


SNAPSHOTS = {s: build_snapshot(s, MARKET[s], NOW, CONFIG) for s in CONFIG.symbols}
print(json.dumps(SNAPSHOTS["AAPL"]["indicators"], indent=2))

{
  "latest": {
    "bar_open": "2026-09-25T14:30:00-04:00",
    "close": 218.94,
    "ema20": 217.93,
    "ema50": 217.88,
    "rsi14": 57.94,
    "atr14": 1.29,
    "volume": 1077700.0,
    "volume_sma20": 958845.0
  },
  "previous": {
    "bar_open": "2026-09-25T13:30:00-04:00",
    "close": 219.86,
    "ema20": 217.83,
    "ema50": 217.84,
    "rsi14": 64.27,
    "atr14": 1.32,
    "volume": 1080300.0,
    "volume_sma20": 954380.0
  }
}


The strategy rules are mechanical, so code can evaluate them too. These two functions are the ground truth the risk engine holds Claude to.

In [8]:
def entry_signal(snapshot: dict, cfg: TradingConfig) -> dict[str, bool]:
    cur = snapshot["indicators"]["latest"]
    prev = snapshot["indicators"]["previous"]
    return {
        "price_above_ema50": cur["close"] > cur["ema50"],
        "ema20_crossed_above_ema50": prev["ema20"] <= prev["ema50"] and cur["ema20"] > cur["ema50"],
        "rsi_in_range": cfg.rsi_min <= cur["rsi14"] <= cfg.rsi_max,
        "volume_above_average": cur["volume"] > cur["volume_sma20"],
    }


def exit_signal(snapshot: dict, position) -> list[str]:
    cur = snapshot["indicators"]["latest"]
    last_bar = snapshot["recent_bars"][-1]
    triggers = []
    if cur["close"] < cur["ema20"]:
        triggers.append(f"close {cur['close']} below 20 EMA {cur['ema20']}")
    if last_bar["low"] <= position.stop_loss:
        triggers.append(f"stop {position.stop_loss} hit")
    if last_bar["high"] >= position.take_profit:
        triggers.append(f"target {position.take_profit} hit")
    return triggers


pd.DataFrame(
    {
        s: {**entry_signal(snap, CONFIG), "rsi14": snap["indicators"]["latest"]["rsi14"]}
        for s, snap in SNAPSHOTS.items()
    }
).T

,price_above_ema50,ema20_crossed_above_ema50,rsi_in_range,volume_above_average,rsi14
SPY,False,False,False,False,21.66
QQQ,True,False,True,True,53.47
AAPL,True,True,True,True,57.94
MSFT,True,True,False,True,69.73


Only AAPL passes all four checks. MSFT has the fresh cross and the volume, but its RSI is above 65. That near miss is the case where a model reasoning in prose is most likely to talk itself into a trade.

## 4. Account state and circuit breakers

The account holds one position: QQQ, bought on Wednesday and now slightly underwater. The circuit breakers run before the model is called. If a hard limit is already breached, there's no reason to spend tokens asking Claude whether to trade.

In production, `day_start_equity` and `peak_equity` must come from the broker or durable storage, not from memory in the process that is doing the trading.

In [9]:
@dataclass
class Position:
    symbol: str
    quantity: float
    entry_price: float
    stop_loss: float
    take_profit: float
    opened_at: str


@dataclass
class Account:
    equity: float
    cash: float
    day_start_equity: float
    peak_equity: float
    positions: dict[str, Position] = field(default_factory=dict)

    @property
    def daily_pnl_pct(self) -> float:
        return (self.equity - self.day_start_equity) / self.day_start_equity * 100

    @property
    def drawdown_pct(self) -> float:
        return (self.peak_equity - self.equity) / self.peak_equity * 100


def pre_trade_gate(account: Account, cfg: TradingConfig) -> list[str]:
    breaches = []
    if account.daily_pnl_pct <= -cfg.daily_loss_limit_pct:
        breaches.append(
            f"daily loss {account.daily_pnl_pct:.2f}% reached the {cfg.daily_loss_limit_pct:g}% limit"
        )
    if account.drawdown_pct >= cfg.max_drawdown_pct:
        breaches.append(
            f"drawdown {account.drawdown_pct:.2f}% reached the {cfg.max_drawdown_pct:g}% limit; "
            "human review required"
        )
    return breaches


# Bought on Wednesday's 12:30 bar: stop 1.5 x ATR below entry, target just over 2R.
qqq_position = Position("QQQ", 36, 540.78, 535.28, 552.00, "2026-09-23T13:30:00-04:00")
qqq_value = qqq_position.quantity * SNAPSHOTS["QQQ"]["indicators"]["latest"]["close"]

ACCOUNT = Account(
    equity=100_000.00,
    cash=100_000.00 - qqq_value,
    day_start_equity=100_350.00,
    peak_equity=101_200.00,
    positions={"QQQ": qqq_position},
)
print(f"Daily P&L: {ACCOUNT.daily_pnl_pct:.2f}%   Drawdown: {ACCOUNT.drawdown_pct:.2f}%")
print(f"Breaches: {pre_trade_gate(ACCOUNT, CONFIG) or 'none'}")

Daily P&L: -0.35%   Drawdown: 1.19%
Breaches: none


## 5. Build the risk engine

This is the core of the notebook. `enforce_risk_rules` takes whatever Claude proposed and decides what may reach a broker. It follows three principles:

1. **Code owns the arithmetic.** It recomputes the stop from ATR and the size from equity, and it uses its own numbers. Any disagreement with Claude is logged as an adjustment.
2. **Only make trades safer.** A size can shrink but never grow. A trade that fails reward:risk is rejected; the engine won't push the target out to make it pass.
3. **Risk-reducing actions always go through.** Exits are allowed even while trading is halted, and if an exit rule fires on a held position without a `CLOSE` from the model, the engine closes it anyway.

Closing a position doesn't free a slot in the same cycle: the close might not fill.

In [10]:
@dataclass
class Order:
    symbol: str
    side: Literal["BUY", "SELL"]
    quantity: float
    order_type: str
    limit_price: float | None
    stop_loss: float | None
    take_profit: float | None
    risk_amount: float
    source: Literal["model", "risk_engine"]


@dataclass
class RiskReview:
    approved: list[Order] = field(default_factory=list)
    rejected: list[dict] = field(default_factory=list)
    adjustments: list[str] = field(default_factory=list)
    warnings: list[str] = field(default_factory=list)
    halted: bool = False
    human_review: bool = False


def exit_order(symbol: str, quantity: float, source: str) -> Order:
    return Order(symbol, "SELL", quantity, "MARKET", None, None, None, 0.0, source)


def in_session(now: pd.Timestamp, cfg: TradingConfig) -> bool:
    local = now.tz_convert(ET)
    return local.weekday() < 5 and cfg.session_open <= f"{local:%H:%M}" < cfg.session_close


def events_in_blackout(symbol, events, now, cfg) -> list[str]:
    window = pd.Timedelta(minutes=cfg.event_blackout_minutes)
    hits = []
    for event in events:
        when = pd.Timestamp(event["time"])
        applies = "*" in event["symbols"] or symbol in event["symbols"]
        if applies and abs(when - now) <= window:
            hits.append(f"{event['name']} at {when.tz_convert(ET):%H:%M} ET")
    return hits


def size_position(entry, stop, equity, cash_available, cfg) -> tuple[float, str]:
    risk_budget = equity * cfg.risk_per_trade_pct / 100
    position_budget = equity * cfg.max_position_pct / 100
    buying_power = cash_available * cfg.max_leverage
    limits = {
        f"{cfg.risk_per_trade_pct:g}% risk per trade": risk_budget / (entry - stop),
        f"{cfg.max_position_pct:g}% position cap": position_budget / entry,
        "available cash": buying_power / entry,
    }
    binding = min(limits, key=limits.get)
    qty = limits[binding] if cfg.fractional_quantity else math.floor(limits[binding])
    return max(qty, 0), binding


def review_buy(action, snapshot, account, events, now, cfg, open_count, cash_available):
    # Returns (order, rejection reasons, adjustments).
    if snapshot is None or not snapshot["data_quality"]["ok"]:
        issues = snapshot["data_quality"]["issues"] if snapshot else ["no data"]
        return None, ["data quality: " + "; ".join(issues)], []

    reasons = []
    if not in_session(now, cfg):
        reasons.append("outside trading hours")
    if action.symbol in account.positions:
        reasons.append("already holding this symbol (no adding, no averaging down)")
    if open_count >= cfg.max_open_positions:
        reasons.append(f"already at the max of {cfg.max_open_positions} open positions")
    if action.confidence == "LOW":
        reasons.append("LOW confidence: when in doubt, don't trade")
    if blackout := events_in_blackout(action.symbol, events, now, cfg):
        reasons.append(f"within {cfg.event_blackout_minutes} min of " + ", ".join(blackout))
    failed = [name for name, ok in entry_signal(snapshot, cfg).items() if not ok]
    if failed:
        reasons.append("entry conditions not met in code: " + ", ".join(failed))

    latest = snapshot["indicators"]["latest"]
    last_close = latest["close"]
    entry = action.entry_price if action.order_type == "LIMIT" else last_close
    deviation_pct = abs(entry - last_close) / last_close * 100
    if deviation_pct > cfg.max_entry_deviation_pct:
        reasons.append(
            f"entry {entry} is {deviation_pct:.1f}% away from the last close {last_close}"
        )

    stop = round(entry - cfg.atr_stop_multiple * latest["atr14"], 2)
    risk_per_share = entry - stop
    reward_risk = (action.take_profit - entry) / risk_per_share
    if reward_risk < cfg.min_reward_risk:
        reasons.append(
            f"reward:risk {reward_risk:.2f} is below {cfg.min_reward_risk:g} "
            f"(entry {entry}, stop {stop}, target {action.take_profit})"
        )

    max_qty, binding = size_position(entry, stop, account.equity, cash_available, cfg)
    requested = action.quantity if cfg.fractional_quantity else math.floor(action.quantity)
    qty = min(requested, max_qty)
    if qty <= 0:
        reasons.append("position size rounds to zero")
    if reasons:
        return None, reasons, []

    adjustments = []
    if not math.isclose(action.stop_loss, stop, abs_tol=0.01):
        adjustments.append(
            f"stop {action.stop_loss} -> {stop} ({cfg.atr_stop_multiple:g} x ATR {latest['atr14']} below entry)"
        )
    if qty < action.quantity:
        adjustments.append(f"quantity {action.quantity:g} -> {qty:g} (binding limit: {binding})")
    limit_price = entry if action.order_type == "LIMIT" else None
    order = Order(
        symbol=action.symbol,
        side="BUY",
        quantity=qty,
        order_type=action.order_type,
        limit_price=limit_price,
        stop_loss=stop,
        take_profit=action.take_profit,
        risk_amount=round(qty * risk_per_share, 2),
        source="model",
    )
    return order, [], adjustments


def _reject(review: RiskReview, action: TradeAction, *reasons: str) -> None:
    review.rejected.append(
        {"symbol": action.symbol, "action": action.action, "reasons": list(reasons)}
    )


def enforce_risk_rules(decision, account, snapshots, events, now, cfg=CONFIG) -> RiskReview:
    review = RiskReview()
    breaches = pre_trade_gate(account, cfg)
    review.halted = bool(breaches)
    review.human_review = account.drawdown_pct >= cfg.max_drawdown_pct
    review.warnings.extend(breaches)

    # The model's account figures are informational. The broker's figures win.
    for name, broker_value in [
        ("equity", account.equity),
        ("daily_pnl_pct", account.daily_pnl_pct),
        ("drawdown_pct", account.drawdown_pct),
    ]:
        model_value = getattr(decision.account_status, name)
        if not math.isclose(model_value, broker_value, rel_tol=0.01, abs_tol=0.05):
            review.warnings.append(
                f"model reported {name}={model_value:g}, broker says {broker_value:.2f}"
            )

    open_count = len(account.positions)
    cash_available = account.cash
    seen, closing = set(), set()
    for action in decision.actions:
        if action.symbol not in cfg.symbols:
            _reject(review, action, "not in the tradable universe")
            continue
        if action.symbol in seen:
            _reject(review, action, "duplicate action for this symbol")
            continue
        seen.add(action.symbol)
        position = account.positions.get(action.symbol)

        if action.action == "SELL":
            _reject(review, action, "shorts are disabled; exits must use CLOSE")
        elif action.action == "CLOSE":
            if position is None:
                _reject(review, action, "no open position to close")
            else:
                closing.add(action.symbol)
                review.approved.append(exit_order(action.symbol, position.quantity, "model"))
        elif action.action == "HOLD" and position is None:
            review.warnings.append(
                f"{action.symbol}: HOLD with no open position (treated as NO_TRADE)"
            )
        elif action.action == "BUY":
            if review.halted:
                _reject(review, action, *breaches)
                continue
            order, reasons, adjustments = review_buy(
                action,
                snapshots.get(action.symbol),
                account,
                events,
                now,
                cfg,
                open_count,
                cash_available,
            )
            if order is None:
                _reject(review, action, *reasons)
                continue
            review.approved.append(order)
            review.adjustments.extend(f"{action.symbol}: {a}" for a in adjustments)
            open_count += 1
            cash_available -= order.quantity * (
                order.limit_price or snapshots[action.symbol]["indicators"]["latest"]["close"]
            )

    # Exits reduce risk, so the engine enforces them even when the model didn't.
    for symbol, position in account.positions.items():
        if symbol in closing:
            continue
        snapshot = snapshots.get(symbol)
        if snapshot is None or not snapshot["data_quality"]["ok"]:
            review.warnings.append(
                f"{symbol}: can't evaluate exit on bad data; the broker-side stop still applies"
            )
        elif triggers := exit_signal(snapshot, position):
            review.approved.append(exit_order(symbol, position.quantity, "risk_engine"))
            review.warnings.append(
                f"{symbol}: exit rule fired ({'; '.join(triggers)}) with no CLOSE; risk engine closed it"
            )

    missing = [s for s in cfg.symbols if s not in seen]
    if missing:
        review.warnings.append(f"no decision for {', '.join(missing)}; treated as NO_TRADE")
    return review

A small printer makes each review readable at a glance:

In [11]:
def print_review(decision: TradingDecision, review: RiskReview, source: str) -> None:
    print(f"Decision source: {source}   as_of: {decision.timestamp}")
    for a in decision.actions:
        numbers = f" qty={a.quantity:g} entry={a.entry_price:g} stop={a.stop_loss:g} target={a.take_profit:g}"
        print(
            f"  {a.symbol:<5} {a.action:<8} {a.confidence:<6}{numbers if a.action == 'BUY' else ''}"
        )
        print(f"        {a.reasoning}")
    for w in decision.warnings:
        print(f"  decision warning: {w}")

    status = "HALTED" if review.halted else "active"
    print(f"\nRisk engine ({status}{', HUMAN REVIEW REQUIRED' if review.human_review else ''})")
    for o in review.approved:
        detail = f" limit={o.limit_price} stop={o.stop_loss} target={o.take_profit} risk=${o.risk_amount:,.2f}"
        print(
            f"  APPROVED  {o.side} {o.quantity:g} {o.symbol} {o.order_type}"
            f"{detail if o.side == 'BUY' else ''} [{o.source}]"
        )
    for r in review.rejected:
        print(f"  REJECTED  {r['action']} {r['symbol']}: {'; '.join(r['reasons'])}")
    for adj in review.adjustments:
        print(f"  ADJUSTED  {adj}")
    for w in review.warnings:
        print(f"  WARNING   {w}")
    if not (review.approved or review.rejected):
        print("  no orders")

## 6. Red-team the risk engine (no API call)

Before trusting the engine with Claude's output, feed it a proposal that breaks the rules on purpose. Each action below exercises a different check:

- **AAPL:** a valid setup, but 400 shares (far over the limits) and a stop tighter than 1.5 × ATR
- **MSFT:** a confident `BUY` on the RSI near miss
- **SPY:** a `BUY` at an invented price, with `LOW` confidence, on a downtrend
- **QQQ:** a `SELL` where the model meant "exit"
- **TSLA:** a symbol outside the universe

In [12]:
def buy(symbol, qty, entry, stop, target, confidence, reasoning):
    return TradeAction(
        symbol=symbol,
        action="BUY",
        quantity=qty,
        order_type="LIMIT",
        entry_price=entry,
        stop_loss=stop,
        take_profit=target,
        risk_amount=round(qty * (entry - stop), 2),
        confidence=confidence,
        reasoning=reasoning,
    )


bad_decision = TradingDecision(
    timestamp=NOW.isoformat(),
    account_status=AccountStatus(equity=100_000, daily_pnl_pct=-0.35, drawdown_pct=1.19),
    actions=[
        buy("AAPL", 400, 218.94, 217.80, 226.00, "HIGH", "Golden cross with confirming volume."),
        buy("MSFT", 40, 469.81, 465.49, 479.00, "HIGH", "Golden cross; momentum is strong."),
        buy("SPY", 30, 540.00, 535.00, 555.00, "LOW", "Oversold bounce likely."),
        TradeAction(
            symbol="QQQ",
            action="SELL",
            quantity=36,
            order_type="MARKET",
            entry_price=537.83,
            stop_loss=0,
            take_profit=0,
            risk_amount=0,
            confidence="HIGH",
            reasoning="Close below the 20 EMA; exit.",
        ),
        buy("TSLA", 10, 250.00, 245.00, 265.00, "MEDIUM", "Not in the universe."),
    ],
    warnings=[],
)

print_review(
    bad_decision, enforce_risk_rules(bad_decision, ACCOUNT, SNAPSHOTS, [], NOW), "red-team"
)

Decision source: red-team   as_of: 2026-09-25T15:35:00-04:00
  AAPL  BUY      HIGH   qty=400 entry=218.94 stop=217.8 target=226
        Golden cross with confirming volume.
  MSFT  BUY      HIGH   qty=40 entry=469.81 stop=465.49 target=479
        Golden cross; momentum is strong.
  SPY   BUY      LOW    qty=30 entry=540 stop=535 target=555
        Oversold bounce likely.
  QQQ   SELL     HIGH  
        Close below the 20 EMA; exit.
  TSLA  BUY      MEDIUM qty=10 entry=250 stop=245 target=265
        Not in the universe.

Risk engine (active)
  APPROVED  BUY 91 AAPL LIMIT limit=218.94 stop=217.0 target=226.0 risk=$176.54 [model]
  APPROVED  SELL 36 QQQ MARKET [risk_engine]
  REJECTED  BUY MSFT: entry conditions not met in code: rsi_in_range
  REJECTED  BUY SPY: LOW confidence: when in doubt, don't trade; entry conditions not met in code: price_above_ema50, ema20_crossed_above_ema50, rsi_in_range, volume_above_average; entry 540.0 is 7.2% away from the last close 503.74
  REJECTED  SELL

Every rule-breaking action was caught, and each rejection names the rule it broke. A few things are worth pointing out:

- **AAPL** was approved, but on the engine's terms. The stop moved to the ATR rule, and the size shrank from 400 shares to what the limits allow.
- **The binding limit on AAPL was the 20% position cap, not the 1% risk rule.** With a 1.5 × ATR stop on hourly bars, the stop is only about 1% below the entry, so 1% of equity at risk would mean a position larger than the whole account. The cap keeps the real risk on this trade near 0.2% of equity. That's worth knowing when you pick the values for your own limits.
- **QQQ:** the `SELL` was rejected as a short, and the engine then closed the position anyway because the exit rule fired. A wrong action label didn't leave a position unmanaged.

## 7. Wire up the cycle

Four functions tie the pipeline together:

- `propose_trades` is the only function that calls Claude. Any failure (no credentials, a network or API error, a refusal, a truncated response, or output that fails validation) returns `None` plus a reason instead of raising.
- `fail_closed_decision` builds a decision in code when Claude isn't consulted or can't answer: hold open positions and open nothing.
- `build_model_input` assembles the account, snapshots, events, and headlines into the payload Claude sees.
- `run_cycle` runs the circuit breakers, then Claude or the fail-closed path, then the risk engine, and appends an audit record.

In [13]:
AUDIT_LOG: list[dict] = []


def fail_closed_decision(account: Account, now, cfg: TradingConfig, reason: str) -> TradingDecision:
    # A code-generated decision: hold open positions (broker-side stops still apply), trade nothing.
    actions = []
    for symbol in cfg.symbols:
        position = account.positions.get(symbol)
        actions.append(
            TradeAction(
                symbol=symbol,
                action="HOLD" if position else "NO_TRADE",
                quantity=position.quantity if position else 0,
                order_type="MARKET",
                entry_price=0,
                stop_loss=position.stop_loss if position else 0,
                take_profit=position.take_profit if position else 0,
                risk_amount=0,
                confidence="HIGH",
                reasoning=reason,
            )
        )
    status = AccountStatus(
        equity=account.equity,
        daily_pnl_pct=round(account.daily_pnl_pct, 2),
        drawdown_pct=round(account.drawdown_pct, 2),
    )
    return TradingDecision(
        timestamp=now.isoformat(), account_status=status, actions=actions, warnings=[reason]
    )


def build_model_input(account, snapshots, events, headlines, now) -> dict:
    return {
        "as_of": now.isoformat(),
        "account": {
            "equity": account.equity,
            "cash": round(account.cash, 2),
            "daily_pnl_pct": round(account.daily_pnl_pct, 2),
            "drawdown_pct": round(account.drawdown_pct, 2),
            "open_positions": [asdict(p) for p in account.positions.values()],
        },
        "market": list(snapshots.values()),
        "scheduled_events": events,
        "headlines": headlines,
    }


def propose_trades(model_input: dict) -> tuple[TradingDecision | None, str | None]:
    # Any failure returns (None, reason) so the caller can fail closed.
    if not (client.api_key or client.auth_token):
        return None, "no Anthropic credentials; set ANTHROPIC_API_KEY"
    try:
        response = client.messages.parse(
            model=MODEL,
            max_tokens=16000,
            system=SYSTEM_PROMPT,
            thinking={"type": "adaptive"},
            messages=[
                {
                    "role": "user",
                    "content": "This cycle's input:\n\n" + json.dumps(model_input, indent=2),
                }
            ],
            output_format=TradingDecision,
        )
    except anthropic.APIConnectionError as e:
        return None, f"connection error: {e}"
    except anthropic.APIStatusError as e:
        return None, f"API error {e.status_code}: {e.message}"
    except pydantic.ValidationError as e:
        return None, f"response failed schema validation ({e.error_count()} errors)"
    if response.stop_reason != "end_turn" or response.parsed_output is None:
        return None, f"no usable decision (stop_reason={response.stop_reason})"
    return response.parsed_output, None


def run_cycle(account, market, events, headlines, now, cfg=CONFIG):
    snapshots = {s: build_snapshot(s, market[s], now, cfg) for s in cfg.symbols}
    model_input = build_model_input(account, snapshots, events, headlines, now)

    if breaches := pre_trade_gate(account, cfg):
        source = "circuit_breaker"
        decision = fail_closed_decision(
            account, now, cfg, "Hard limit breached: " + "; ".join(breaches)
        )
    else:
        source = "model"
        decision, error = propose_trades(model_input)
        if decision is None:
            source = "fail_closed"
            decision = fail_closed_decision(
                account, now, cfg, f"Model unavailable ({error}); failing closed"
            )

    review = enforce_risk_rules(decision, account, snapshots, events, now, cfg)
    AUDIT_LOG.append(
        {
            "as_of": now.isoformat(),
            "source": source,
            "model": MODEL if source == "model" else None,
            "input_sha256": hashlib.sha256(
                json.dumps(model_input, sort_keys=True).encode()
            ).hexdigest(),
            "decision": decision.model_dump(),
            "review": asdict(review),
        }
    )
    print_review(decision, review, source)
    return decision, review

## 8. Trip the circuit breakers (no API call)

Here are two accounts that have hit hard limits. Neither cycle calls Claude. The engine still enforces the QQQ exit, because exits reduce risk.

In [14]:
daily_loss_account = Account(
    equity=97_000.00,
    cash=97_000.00 - qqq_value,
    day_start_equity=100_350.00,
    peak_equity=101_200.00,
    positions={"QQQ": qqq_position},
)
drawdown_account = Account(
    equity=100_000.00,
    cash=100_000.00 - qqq_value,
    day_start_equity=100_350.00,
    peak_equity=112_000.00,
    positions={"QQQ": qqq_position},
)

print("=== Daily loss limit ===")
run_cycle(daily_loss_account, MARKET, [], [], NOW)
print("\n=== Max drawdown ===")
_ = run_cycle(drawdown_account, MARKET, [], [], NOW)

=== Daily loss limit ===
Decision source: circuit_breaker   as_of: 2026-09-25T15:35:00-04:00
  SPY   NO_TRADE HIGH  
        Hard limit breached: daily loss -3.34% reached the 3% limit
  QQQ   HOLD     HIGH  
        Hard limit breached: daily loss -3.34% reached the 3% limit
  AAPL  NO_TRADE HIGH  
        Hard limit breached: daily loss -3.34% reached the 3% limit
  MSFT  NO_TRADE HIGH  
        Hard limit breached: daily loss -3.34% reached the 3% limit
  decision warning: Hard limit breached: daily loss -3.34% reached the 3% limit

Risk engine (HALTED)
  APPROVED  SELL 36 QQQ MARKET [risk_engine]
  WARNING   daily loss -3.34% reached the 3% limit
  WARNING   QQQ: exit rule fired (close 537.83 below 20 EMA 538.7) with no CLOSE; risk engine closed it

=== Max drawdown ===
Decision source: circuit_breaker   as_of: 2026-09-25T15:35:00-04:00
  SPY   NO_TRADE HIGH  
        Hard limit breached: drawdown 10.71% reached the 10% limit; human review required
  QQQ   HOLD     HIGH  
        H

## 9. Run a live cycle with Claude

Now the real thing: the healthy account, the four snapshots, a scheduled event that is days away, and a couple of headlines. `propose_trades` sends everything to Claude with adaptive thinking and the `TradingDecision` schema.

Compare Claude's decisions with the code's signal table from step 3. By the rules, the right answer is a `BUY` for AAPL, `NO_TRADE` for MSFT and SPY, and a `CLOSE` for QQQ. Whatever Claude proposes, the risk engine checks it. If the call fails for any reason (no key, a network error, a refusal, or output that doesn't validate), the cycle fails closed: it holds positions, opens nothing, and the engine still enforces exits.

> The saved outputs for this step and the next come from a run without an API key, so they show the fail-closed path. Run the notebook with your key to see Claude's decisions.

In [15]:
EVENTS = [{"time": "2026-10-02T08:30:00-04:00", "name": "Jobs report", "symbols": ["*"]}]
HEADLINES = [
    {
        "time": "2026-09-25T13:10:00-04:00",
        "headline": "Large-cap tech leads a quiet afternoon session",
    },
    {
        "time": "2026-09-25T11:45:00-04:00",
        "headline": "Treasury yields little changed ahead of next week's data",
    },
]

decision, review = run_cycle(ACCOUNT, MARKET, EVENTS, HEADLINES, NOW)

Decision source: fail_closed   as_of: 2026-09-25T15:35:00-04:00
  SPY   NO_TRADE HIGH  
        Model unavailable (no Anthropic credentials; set ANTHROPIC_API_KEY); failing closed
  QQQ   HOLD     HIGH  
        Model unavailable (no Anthropic credentials; set ANTHROPIC_API_KEY); failing closed
  AAPL  NO_TRADE HIGH  
        Model unavailable (no Anthropic credentials; set ANTHROPIC_API_KEY); failing closed
  MSFT  NO_TRADE HIGH  
        Model unavailable (no Anthropic credentials; set ANTHROPIC_API_KEY); failing closed
  decision warning: Model unavailable (no Anthropic credentials; set ANTHROPIC_API_KEY); failing closed

Risk engine (active)
  APPROVED  SELL 36 QQQ MARKET [risk_engine]
  WARNING   QQQ: exit rule fired (close 537.83 below 20 EMA 538.7) with no CLOSE; risk engine closed it


## 10. Run a messy cycle: bad data and an earnings blackout

Real feeds break. This cycle corrupts two feeds and schedules AAPL earnings 25 minutes out:

- **SPY:** the latest bar has its high and low swapped and zero volume.
- **MSFT:** the feed stalled, and the latest bar is from yesterday.
- **AAPL:** the setup is still valid, but earnings land inside the 30-minute blackout.

The data-quality report tells Claude what's wrong. Whatever Claude decides, the engine rejects a `BUY` on bad data or inside a blackout.

In [16]:
messy_market = {s: bars.copy() for s, bars in MARKET.items()}
spy_last = messy_market["SPY"].index[-1]
messy_market["SPY"].loc[spy_last, ["high", "low"]] = (
    messy_market["SPY"].loc[spy_last, ["low", "high"]].values
)
messy_market["SPY"].loc[spy_last, "volume"] = 0
messy_market["MSFT"] = messy_market["MSFT"].iloc[: -len(BAR_OPENS)]  # drop today's bars

messy_snapshots = {s: build_snapshot(s, messy_market[s], NOW, CONFIG) for s in CONFIG.symbols}
for s in ("SPY", "MSFT"):
    print(s, messy_snapshots[s]["data_quality"])

messy_events = EVENTS + [
    {"time": "2026-09-25T16:00:00-04:00", "name": "AAPL earnings", "symbols": ["AAPL"]}
]
print()
_ = run_cycle(ACCOUNT, messy_market, messy_events, HEADLINES, NOW)

SPY {'ok': False, 'issues': ["1 recent bar(s) where high/low don't contain open/close", 'zero volume in recent bars']}
MSFT {'ok': False, 'issues': ['stale data: last bar closed 1445 min ago']}

Decision source: fail_closed   as_of: 2026-09-25T15:35:00-04:00
  SPY   NO_TRADE HIGH  
        Model unavailable (no Anthropic credentials; set ANTHROPIC_API_KEY); failing closed
  QQQ   HOLD     HIGH  
        Model unavailable (no Anthropic credentials; set ANTHROPIC_API_KEY); failing closed
  AAPL  NO_TRADE HIGH  
        Model unavailable (no Anthropic credentials; set ANTHROPIC_API_KEY); failing closed
  MSFT  NO_TRADE HIGH  
        Model unavailable (no Anthropic credentials; set ANTHROPIC_API_KEY); failing closed
  decision warning: Model unavailable (no Anthropic credentials; set ANTHROPIC_API_KEY); failing closed

Risk engine (active)
  APPROVED  SELL 36 QQQ MARKET [risk_engine]
  WARNING   QQQ: exit rule fired (close 537.83 below 20 EMA 538.7) with no CLOSE; risk engine closed it


To check the engine without relying on Claude's judgment, replay step 6's red-team proposal against these messy inputs. The AAPL `BUY` that passed in step 6 is now blocked by the earnings blackout, and the `BUY`s on bad feeds are blocked by the data checks before any other rule runs.

In [17]:
replay = enforce_risk_rules(bad_decision, ACCOUNT, messy_snapshots, messy_events, NOW)
for r in replay.rejected:
    print(f"REJECTED  {r['action']} {r['symbol']}: {'; '.join(r['reasons'])}")
print(f"Approved: {[f'{o.side} {o.quantity:g} {o.symbol}' for o in replay.approved]}")

REJECTED  BUY AAPL: within 30 min of AAPL earnings at 16:00 ET
REJECTED  BUY MSFT: data quality: stale data: last bar closed 1445 min ago
REJECTED  BUY SPY: data quality: 1 recent bar(s) where high/low don't contain open/close; zero volume in recent bars
REJECTED  SELL QQQ: shorts are disabled; exits must use CLOSE
REJECTED  BUY TSLA: not in the tradable universe
Approved: ['SELL 36 QQQ']


## 11. Review the audit trail

Every cycle appended one record: what went in (hashed), who decided (the model, a circuit breaker, or the fail-closed path), what was proposed, and what the engine did with it. In production, write these records to append-only storage. They are how you answer "why did it do that?" a week later.

In [18]:
print(f"{len(AUDIT_LOG)} cycles logged")
for entry in AUDIT_LOG:
    review = entry["review"]
    print(
        f"  {entry['as_of']}  source={entry['source']:<15} approved={len(review['approved'])} "
        f"rejected={len(review['rejected'])} halted={review['halted']} input={entry['input_sha256'][:12]}"
    )

print(json.dumps(AUDIT_LOG[-1]["review"], indent=2))

4 cycles logged
  2026-09-25T15:35:00-04:00  source=circuit_breaker approved=1 rejected=0 halted=True input=012ce1108144
  2026-09-25T15:35:00-04:00  source=circuit_breaker approved=1 rejected=0 halted=True input=4877222ee625
  2026-09-25T15:35:00-04:00  source=fail_closed     approved=1 rejected=0 halted=False input=60bcb074f553
  2026-09-25T15:35:00-04:00  source=fail_closed     approved=1 rejected=0 halted=False input=ddba576a072d
{
  "approved": [
    {
      "symbol": "QQQ",
      "side": "SELL",
      "quantity": 36,
      "order_type": "MARKET",
      "limit_price": null,
      "stop_loss": null,
      "take_profit": null,
      "risk_amount": 0.0,
      "source": "risk_engine"
    }
  ],
  "rejected": [],
  "adjustments": [],
  "warnings": [
    "QQQ: exit rule fired (close 537.83 below 20 EMA 538.7) with no CLOSE; risk engine closed it"
  ],
  "halted": false,
  "human_review": false
}


## Where to go from here

- **Paper trade and replay before anything else.** Run the pipeline over months of historical cycles and compare Claude's decisions with the code's signals. Every disagreement is either a bug in your prompt or a case where judgment added something. Both are worth reading.
- **Put the protection at the broker.** Submit every entry as a bracket order, with the stop and target attached, so the position stays protected even if this loop stops running.
- **Add a human approval step** between `review.approved` and order submission until you trust the system. `review.human_review` already tells you when a person must look.
- **Decide what Claude is for.** This strategy is fully mechanical, so code alone could trade it. Claude earns its place on the inputs code can't judge well: headlines, event context, odd-looking data, and conflicting signals. As you add discretionary rules to the prompt, keep every hard limit in the risk engine.